<a href="https://colab.research.google.com/github/Lagnajit09/100x_AI_ML/blob/main/SystemOneModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Classification by Logit-Slicing: turning a text generator into a decision-maker without training**


---



---



### The plan in plain words:

- Glue the question + the four boxes + the text into one input string.
- Run the model once (your prefill — the part before the decode loop).
- Look at the logits at the very last position (your logits[0, -1]).
- Out of 49,000 vocab logits, pull out just the four that correspond to "A", "B", "C", "D".
- Softmax those four. Done. Four probabilities, one per box.

No loop.

### Step 1 — Build the input

In [ ]:
state   = "Pikachu"
options = ["electric", "grass", "fire", "water"]

letters = ["A", "B", "C", "D"]
block   = "\n".join(f"{L}. {o}" for L, o in zip(letters, options))

prompt = f"""What type is this Pokemon?

{state}

A. electric
B. grass
C. fire
D. water

Answer:"""

**Difference from LLMs**. Earlier the prompt was "Once upon a time" — just the thing to continue. Here the prompt contains the question, the four boxes, and the thing to judge, all glued together. This is the "everything goes in as one input" idea, now as an actual string.

The prompt ends on "Answer:" deliberately. We're setting the model up so the very next token it would produce is a letter. We're not going to let it produce anything — we're going to peek at what it would have said.

### Step 2 — Tokenize

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tok   = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M")
model.eval()

inputs = tok(prompt, return_tensors="pt")   # {'input_ids': (1, T), 'attention_mask': (1, T)}

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

### Step 3 — One forward pass

In [ ]:
with torch.no_grad():
    out = model(**inputs)
    logits = out.logits          # (1, T, vocab_size)

### Step 4 — The key step: grab only the box logits

In [ ]:
last_logits = logits[0, -1]      # (vocab_size,)  — the vote at the "Answer:" position

# token id for each letter (with a leading space, the way it appears after "Answer:")
option_ids = [tok.encode(f" {L}")[-1] for L in letters]   # e.g. [330, 396, 356, 413]

option_logits = last_logits[option_ids]    # (4,) — pull out just those four

**This is the entire trick.**

In _sample() function we did top-p: take the full vocab distribution, keep a slice, throw the rest away. This is the same move pushed to the extreme. Instead of keeping the nucleus, we keep exactly four tokens — the ones that spell your answer letters — and ignore the other ~49,000.

Line by line:

- `logits[0, -1]` — `logits[0, -1]` from `generate()`. The distribution at the final position.
- `tok.encode(" A")[-1]` — asks the tokenizer "what's the id for the token A?" It's one integer. We get four integers, one per letter.
- `last_logits[option_ids]` — fancy indexing. "From the 49,000 numbers, give me the four at these positions." Result: four raw scores.

### Step 5 — Softmax → one probability per box

In [ ]:
probs = torch.softmax(option_logits, dim=-1)     # (4,) sums to 1.0

for o, p in zip(options, probs.tolist()):
    print(f"{o:10s} {p:.3f}")

electric   0.249
grass      0.220
fire       0.281
water      0.249


**How the probability "spreads to each box" — the trace**

Here's the flow with illustrative numbers. Watch a single number turn into four:

```
                    prompt ends: "...D. water\n\nAnswer:"
                                      │
                        one forward pass through SmolLM
                                      │
                                      ▼
        last_logits  =  (49152 raw scores, one per vocab token)
                          ...
                          " A"  →  8.2      ← electric
                          " B"  →  4.1      ← grass
                          " C"  →  5.0      ← fire
                          " D"  →  4.7      ← water
                          " the"→ 9.9       ← model actually likes this most, we ignore it
                          ...
                                      │
             slice out ONLY the four letter positions
                                      │
                                      ▼
        option_logits = [8.2, 4.1, 5.0, 4.7]
                                      │
                                    softmax
                                      │  e^8.2=3641, e^4.1=60, e^5.0=148, e^4.7=110
                                      │  sum = 3959
                                      ▼
        probs = [3641/3959, 60/3959, 148/3959, 110/3959]
              = [0.920,     0.015,   0.037,    0.028]
                electric    grass    fire      water
```

## Full Code Block with **Qwen2.5-3B-Instruct**:

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# The ONLY change from the previous script: a bigger, instruction-tuned model.
MODEL = "Qwen/Qwen2.5-3B-Instruct"      # try also: "meta-llama/Llama-3.2-3B-Instruct"

tok   = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="auto"
)
model.eval()


def classify(state, options, question="Pick the best matching label."):
    letters = [chr(ord("A") + i) for i in range(len(options))]
    block   = "\n".join(f"{L}. {o}" for L, o in zip(letters, options))

    # Instruction-tuned models expect the chat template, not a raw string.
    # (This is the one real addition vs the SmolLM version -- we wrap the
    #  prompt the way the model was fine-tuned to receive it.)
    user_msg = f"{question}\n\nText:\n{state}\n\nOptions:\n{block}\n\nReply with only the letter."
    prompt = tok.apply_chat_template(
        [{"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True,
    )

    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        last_logits = model(**inputs).logits[0, -1]      # (vocab_size,)

    option_ids    = [tok.encode(f" {L}", add_special_tokens=False)[-1] for L in letters]
    option_logits = last_logits[option_ids]
    probs         = torch.softmax(option_logits.float(), dim=-1)
    return dict(zip(options, probs.tolist()))



config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
if __name__ == "__main__":
    print("--- Pokemon typing ---")
    for o, p in classify("Pikachu", ["fire", "grass", "electric", "water"],
                         question="What type is this Pokemon?").items():
        print(f"  {o:10s} {p:.3f}")

    print("\n--- Support ticket routing ---")
    ticket = "Hi, we were billed twice for March. Please refund the duplicate."
    for o, p in classify(ticket, ["billing", "technical", "sales", "other"],
                         question="Which department should handle this?").items():
        print(f"  {o:10s} {p:.3f}")

    print("\n--- Urgency ---")
    for o, p in classify("The app is slow and I also want to upgrade my plan",
         ["billing", "technical", "sales", "other"],
         question="Which department?").items():
        print(f"  {o:10s} {p:.3f}")

--- Pokemon typing ---
  fire       0.000
  grass      0.000
  electric   1.000
  water      0.000

--- Support ticket routing ---
  billing    1.000
  technical  0.000
  sales      0.000
  other      0.000

--- Urgency ---
  billing    0.019
  technical  0.981
  sales      0.000
  other      0.000



---
---

# **Trying Out Laya: Open-Source Non-autoregressive System 1 decision engine**

---
---



### Step 1 — Install

In [ ]:
!pip install laya -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.7/79.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.5/214.5 kB 12.8 MB/s eta 0:00:00


## PART A -- see one full distribution (the honest spread, vs Qwen's 1.000)


---



In [ ]:
import json
from laya import Router

router = Router()

q_dept = {"department": {
    "type": "choice",
    "instructions": "Which department should handle this?",
    "criteria": {"billing": "invoices, payments, refunds",
                 "technical": "bugs, outages, system errors",
                 "other": "everything else"}}}

r = router.predict("Hi, we were billed twice for March. Please refund the duplicate.", q_dept)
print(json.dumps(r["answers"]["department"], indent=2))
print("routing:", r["routing"]["model"])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/router.py:456: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  agent = Agent(repo, **kwargs)


{
  "type": "choice",
  "choice": "billing",
  "probabilities": {
    "billing": 0.9824,
    "technical": 0.0097,
    "other": 0.0079
  },
  "confidence": 0.9084,
  "answer_confidence": 0.9824,
  "action": {
    "act_probability": 1.0
  }
}
routing: english


## PART B -- measure ECE on a tiny labelled set


---



In [ ]:
examples = [
    ("I was charged twice, please refund the extra amount.", "billing"),
    ("The dashboard throws a 500 error every time I log in.", "technical"),
    ("Can you send me a copy of last month's invoice?",       "billing"),
    ("The app crashes on startup after the latest update.",   "technical"),
    ("I'd like to upgrade to the annual plan.",               "other"),
    ("My payment method was declined but I was still billed.","billing"),
    ("Login page is completely blank on Safari.",             "technical"),
    ("How do I change my account email address?",             "other"),
    ("You double-charged my card, I want the duplicate back.","billing"),
    ("Getting a timeout error when uploading files.",         "technical"),
]

records = []   # (confidence, correct) for each prediction
for text, truth in examples:
    ans  = router.predict(text, q_dept)["answers"]["department"]
    pick = ans["choice"]
    conf = ans["confidence"]
    records.append((conf, int(pick == truth)))
    mark = "OK " if pick == truth else "XX "
    print(f"{mark} pred={pick:9s} truth={truth:9s} conf={conf:.3f}  | {text[:45]}")

OK  pred=billing   truth=billing   conf=0.718  | I was charged twice, please refund the extra 
OK  pred=technical truth=technical conf=0.740  | The dashboard throws a 500 error every time I
OK  pred=billing   truth=billing   conf=0.837  | Can you send me a copy of last month's invoic
OK  pred=technical truth=technical conf=0.558  | The app crashes on startup after the latest u
XX  pred=billing   truth=other     conf=0.262  | I'd like to upgrade to the annual plan.
OK  pred=billing   truth=billing   conf=0.852  | My payment method was declined but I was stil
OK  pred=technical truth=technical conf=0.320  | Login page is completely blank on Safari.
XX  pred=technical truth=other     conf=0.039  | How do I change my account email address?
OK  pred=billing   truth=billing   conf=0.529  | You double-charged my card, I want the duplic
OK  pred=technical truth=technical conf=0.329  | Getting a timeout error when uploading files.


## PART C -- ECE, from scratch


---



In [ ]:
def expected_calibration_error(records, n_bins=5):
    """records: list of (confidence, correct 0/1). Returns ECE + per-bin table."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in records:
        idx = min(int(conf * n_bins), n_bins - 1)   # 0.0-0.2 -> bin 0, etc.
        bins[idx].append((conf, correct))

    N = len(records)
    ece = 0.0
    table = []
    for b, bucket in enumerate(bins):
        if not bucket:
            continue
        avg_conf = sum(c for c, _ in bucket) / len(bucket)
        accuracy = sum(y for _, y in bucket) / len(bucket)
        gap      = abs(accuracy - avg_conf)
        weight   = len(bucket) / N
        ece     += weight * gap
        lo, hi   = b / n_bins, (b + 1) / n_bins
        table.append((f"{lo:.1f}-{hi:.1f}", len(bucket), avg_conf, accuracy, gap))
    return ece, table

ece, table = expected_calibration_error(records, n_bins=5)

print("\n=== Reliability table ===")
print(f"{'bucket':10s} {'n':>3s} {'avg_conf':>9s} {'accuracy':>9s} {'gap':>6s}")
for bkt, n, conf, acc, gap in table:
    print(f"{bkt:10s} {n:>3d} {conf:>9.3f} {acc:>9.3f} {gap:>6.3f}")

print(f"\nECE = {ece:.3f}")
print("(Expect nonzero -- shipped Laya is overconfident before temperature fitting.)")


=== Reliability table ===
bucket       n  avg_conf  accuracy    gap
0.0-0.2      1     0.039     0.000  0.039
0.2-0.4      3     0.304     0.667  0.363
0.4-0.6      2     0.544     1.000  0.456
0.6-0.8      2     0.729     1.000  0.271
0.8-1.0      2     0.844     1.000  0.156

ECE = 0.289
(Expect nonzero -- shipped Laya is overconfident before temperature fitting.)
